# 纯 JAX 实现 VMC 梯度计算（修复版）

## 问题分析

原始代码使用 `jax.value_and_grad` 计算梯度，但这与 NetKet 的 `expect_and_grad` 不同。

NetKet 的梯度计算采用 **force-based** 形式：
$$
\nabla \langle E \rangle = \langle (E_{loc} - \langle E \rangle) \nabla \log \psi \rangle
$$

其中 $E_{loc}$ 是局部能量。

## NetKet API 实现路径

参考 `/Users/yangjianfei/mac_vscode/神经网络量子态/4 月/0424/目前的 API 调查进展.md`：

1. `MCState.expect_and_grad` → `expect_and_grad` (dispatch)
2. `get_local_kernel` → `local_value_kernel`
3. `forces_expect_hermitian` → **核心梯度计算函数**
4. 使用 `nkjax.vjp` 计算复数梯度

## 关键修复

**问题**: `flatten_util.ravel_pytree` 在处理 2D 数组时会出错。

**解决方案**: 直接使用 `jax.tree_util.tree_map` 处理 PyTree，避免手动展平。

In [ ]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

In [ ]:
# ===================== 1. H₂ 分子定义 & FCI 基准 =====================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")

In [ ]:
# ===================== 2. NetKet 哈密顿量和采样器 (仅用于生成样本) =====================
ha = nkx.operator.from_pyscf_molecule(mol)

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

print(f"Hilbert space size: {hi.size}")
print(f"Hamiltonian terms: {ha.n_operators}")

In [ ]:
# ===================== 3. 神经网络 Ansatz (Flax NNX) =====================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

In [ ]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

In [ ]:
# ===================== 5. 纯 JAX 实现的 force-based 梯度计算（修复版） =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    
    这对应 NetKet 的 local_value_kernel
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    核心：复刻 NetKet 的 forces_expect_hermitian 函数
    
    使用 force-based 梯度计算：
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    
    关键：对于复数值网络，使用 holomorphic=True
    """
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 计算 ∇log ψ 对每个样本
    # 使用 jax.grad 计算复数梯度（holomorphic=True）
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. 计算 force-based 梯度
    # grad = ⟨(E_loc - E_mean) ∇log ψ⟩ = (1/N) Σ (E_loc[i] - E_mean) ∇log ψ(σ[i])
    # grad_matrix 已经是 PyTree 结构，每个元素的形状是 (n_samples, ...)
    grad = jax.tree_util.tree_map(
        lambda g: jnp.mean(jnp.expand_dims(O_centered, -1) * jnp.conj(g), axis=0),
        grad_matrix
    )
    
    return O_mean, O_std, grad

In [ ]:
# ===================== 6. SR (Stochastic Reconfiguration) 自然梯度 =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_qgt_and_flat_grad(machine, params, sigma, diag_shift=0.1):
    """
    计算量子几何张量 (QGT) / F 矩阵
    
    F = ⟨∇log ψ ∇log ψ†⟩ - ⟨∇log ψ⟩⟨∇log ψ†⟩
    
    这就是 NetKet SR 的核心
    """
    n_samples = sigma.shape[0]
    
    # 计算 ∇log ψ 对每个样本（使用 jax.grad）
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 展平梯度为矩阵
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    # 中心化
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean
    
    # QGT = (1/N) * Σ ∇log ψ ∇log ψ†
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    
    # 正则化
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn


@partial(jax.jit, static_argnames=("machine",))
def apply_sr(machine, params, sigma, grad, diag_shift=0.1):
    """
    应用 SR 预处理：求解 F⁻¹ ∇E
    """
    qgt_reg, unravel_fn = compute_qgt_and_flat_grad(machine, params, sigma, diag_shift)
    
    # 展平梯度
    grad_flat, _ = flatten_util.ravel_pytree(grad)
    
    # 求解线性方程组
    nat_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    
    # 恢复 PyTree 结构
    nat_grad = unravel_fn(nat_grad_flat)
    return nat_grad

In [ ]:
# ===================== 7. 初始化 =====================
rngs = nnx.Rngs(seed=21)
model = SingleStateAnsatz(n_spin_orbitals=hi.size, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

sampler_state = sampler.init_state(machine, params, seed=1)
optimizer = optax.sgd(learning_rate=0.1)
opt_state = optimizer.init(params)

N_ITER = 300
N_SAMPLES = 1008
USE_SR = True

print("初始化完成!")
n_params, _ = flatten_util.ravel_pytree(params)
print(f"模型参数数量：{n_params.shape[0]}")

In [ ]:
# ===================== 8. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (Force-based 梯度 + SR)")
print("="*60)

for step in range(N_ITER):
    # 1. 采样
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=N_SAMPLES // sampler.n_chains
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    
    # 3. 应用 SR 自然梯度
    if USE_SR:
        grad = apply_sr(machine, params, samples, grad, diag_shift=0.1)
    
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 日志
    if step % 20 == 0:
        error = jnp.abs(energy.real - E_fcis[0])
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print("="*60)

## 关键实现总结

### 与 NetKet API 的对应关系

| 本实现 | NetKet 源码 |
|--------|-------------|
| `compute_local_energies` | `local_value_kernel` |
| `forces_expect_hermitian` | `forces_expect_hermitian` |
| `compute_qgt_and_flat_grad` | QGT 计算 |
| `apply_sr` | SR 预处理 |

### 核心公式

1. **局部能量**: $E_{loc}(\sigma) = \sum_\eta H_{\sigma\eta} \frac{\psi(\eta)}{\psi(\sigma)}$

2. **Force-based 梯度**: $\nabla \langle E \rangle = \langle (E_{loc} - \langle E \rangle) \nabla \log \psi \rangle$

3. **QGT 矩阵**: $F = \langle \nabla \log \psi \nabla \log \psi^\dagger \rangle$

4. **自然梯度**: $\nabla_{nat} = F^{-1} \nabla E$

### 关键技术点

1. **复数值网络**: 使用 `jax.grad(..., holomorphic=True)`
2. **PyTree 处理**: 使用 `jax.tree_util.tree_map` 直接处理
3. **向量化**: 使用 `jax.vmap` 批量处理样本

In [ ]:
# ===================== 9. 验证：对比原生 NetKet 结果 =====================
# 重新初始化 NetKet 模型
model_nk = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=nnx.Rngs(21))
vstate = nk.vqs.MCState(sampler, model_nk, n_samples=1008)

print("\n" + "="*60)
print("验证：使用原生 NetKet expect_and_grad")
print("="*60)

# 先运行几步让 sampler 热身
for _ in range(10):
    _ = vstate.sample()

# 计算能量和梯度
stats, grad_nk = vstate.expect_and_grad(ha)
print(f"NetKet 能量：{stats.mean.real:.8f} ± {stats.variance**0.5:.6f} Ha")
print(f"我们计算的：{energy.real:.8f} ± {energy_std:.6f} Ha")